# Transformers preentrenados con HuggingFace

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencia de Datos e IA - IFTS24**

**Asignatura:** Procesamiento de Lenguaje Natural

---

## Objetivo

Usar un transformer preentrenado para análisis de sentimientos en español y, al mismo tiempo, entender qué pasos resume la abstracción de HuggingFace.

## Resultados de aprendizaje

Al final de este notebook vas a poder:

1. Explicar la lógica general de `transfer learning` en PLN.
2. Diferenciar `tokenizer`, modelo y `pipeline`.
3. Ejecutar inferencia con BETO y RoBERTuito sobre frases en español.
4. Comparar la comodidad de una abstracción de alto nivel con lo que ocurre por dentro.
5. Reconocer límites del modelo en ironía, ambigüedad y jerga local.

## Relación con el notebook anterior

En `05` entrenamos una LSTM desde cero. Ahora no vamos a entrenar un modelo nuevo: vamos a reutilizar uno que ya fue preentrenado sobre grandes corpus y que luego fue ajustado para análisis de sentimientos.

## Introducción

Este notebook cambia el paradigma de trabajo. Hasta ahora construimos modelos y los entrenamos sobre corpus pequeños. Con transformers preentrenados, el foco pasa a ser otro: elegir bien el modelo, entender qué abstracción usamos y evaluar cuándo alcanza con inferencia directa y cuándo hace falta fine-tuning.


---

## 1. Instalación de HuggingFace Transformers

En Google Colab, instalamos la librería (ya viene en algunas versiones, pero mejor asegurarnos).

In [ ]:
# Instalamos transformers de HuggingFace
# -q: quiet mode (menos output)
!pip install -q transformers

print("Librería 'transformers' instalada correctamente.")

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import warnings

warnings.filterwarnings('ignore')

torch.set_grad_enabled(False)

print("Librerías importadas.")
print("La primera vez que uses un modelo de HuggingFace, se descargará el tokenizer y los pesos.")


---

## 2. Concepto: Transfer Learning en NLP

### Paradigma tradicional (lo que hicimos en notebooks anteriores):

```
1. Recolectar dataset etiquetado
2. Inicializar pesos aleatoriamente
3. Entrenar desde cero
4. Evaluar
```

**Problemas:**
- Requiere muchos datos (>10k muestras)
- Tiempo de entrenamiento largo
- Recursos computacionales (GPU)
- El modelo aprende lenguaje desde cero

### Paradigma moderno: Transfer Learning

```
1. Tomar modelo pre-entrenado en corpus masivo (Wikipedia, Common Crawl)
2. Adaptar a tu tarea específica (fine-tuning)
3. Requiere pocos datos (<1k muestras)
4. Entrenamiento rápido
```

**Ventajas:**
- El modelo ya "entiende" lenguaje
- Solo ajusta para tu tarea específica
- Funciona con datasets pequeños
- Estado del arte con menos esfuerzo

### Analogía

**Entrenar desde cero:** Como enseñar a leer a alguien desde alfabeto y luego pedirle que clasifique sentimientos

**Transfer learning:** Como pedirle a alguien que ya lee español que aprenda a distinguir reseñas positivas/negativas (mucho más rápido)

### Dos fases del Transfer Learning

**Pre-entrenamiento (ya hecho por otros):**
- Corpus masivo: Wikipedia (3B palabras), Common Crawl (500B palabras)
- Tarea auto-supervisada: Masked Language Modeling (BERT), Next Token Prediction (GPT)
- Meses de entrenamiento en clusters de GPUs
- Costo: $100k - $1M USD

**Fine-tuning (lo que nosotros haremos):**
- Dataset específico: Nuestras reseñas etiquetadas
- Tarea supervisada: Clasificación, NER, QA, etc.
- Horas/días en una GPU
- Costo: $10 - $100 USD

---

## 3. `pipeline`: una abstracción útil, no una caja mágica

Los `pipeline` de HuggingFace son muy cómodos porque encadenan tres pasos en una sola llamada:

1. Tokenización.
2. Paso por el modelo.
3. Post-procesamiento del resultado.

Vamos a usarlos, pero antes vamos a mirar una predicción con más detalle para no perder de vista qué ocurre por adentro.


---

## 4. Cargando BETO: BERT en Español

### ¿Qué es BETO?

**BETO** (Spanish BERT) es un modelo transformer pre-entrenado en español:
- Basado en BERT (Bidirectional Encoder Representations from Transformers)
- Pre-entrenado en Wikipedia en español
- 110M parámetros
- Desarrollado por Universidad de Chile

### Variante que usaremos:

`finiteautomata/beto-sentiment-analysis`:
- BETO fine-tuneado para análisis de sentimientos
- Entrenado en tweets en español
- 3 clases: POS (positivo), NEU (neutral), NEG (negativo)
- Estado del arte para español

In [ ]:
print("Cargando tokenizer y modelo BETO...")
print("La primera ejecución puede tardar unos minutos porque descarga los archivos necesarios.
")

nombre_modelo_beto = "finiteautomata/beto-sentiment-analysis"

tokenizador_beto = AutoTokenizer.from_pretrained(nombre_modelo_beto)
modelo_beto = AutoModelForSequenceClassification.from_pretrained(nombre_modelo_beto)
clasificador = pipeline(
    "sentiment-analysis",
    model=modelo_beto,
    tokenizer=tokenizador_beto
)

print("Modelo BETO cargado correctamente.")
print(f"Nombre: {nombre_modelo_beto}")
print(f"Cantidad de etiquetas: {modelo_beto.config.num_labels}")
print(f"Mapa id2label: {modelo_beto.config.id2label}")


---

## 5. ¿Qué hace el pipeline por dentro?

Antes de usar la interfaz más compacta, vamos a recorrer una predicción en pasos visibles:

1. Tokenizar el texto.
2. Pasar tensores al modelo.
3. Convertir logits en probabilidades.
4. Traducir el índice de mayor probabilidad a una etiqueta legible.


In [ ]:
texto = "Si querés morirte de calor, el lugar está bárbaro, muy recomendable."

print("Predicción paso a paso sin pipeline:")
print("=" * 70)
print(f"Texto: '{texto}'
")

entrada_tokenizada = tokenizador_beto(
    texto,
    return_tensors="pt",
    truncation=True
)

print("Claves devueltas por el tokenizer:")
print(list(entrada_tokenizada.keys()))
print(f"input_ids shape: {entrada_tokenizada['input_ids'].shape}")
print(f"attention_mask shape: {entrada_tokenizada['attention_mask'].shape}")

salida_modelo = modelo_beto(**entrada_tokenizada)
logits = salida_modelo.logits
probabilidades = torch.softmax(logits, dim=1)
indice_predicho = torch.argmax(probabilidades, dim=1).item()
etiqueta_predicha = modelo_beto.config.id2label[indice_predicho]
score_predicho = probabilidades[0][indice_predicho].item()

print("
Resultado del modelo:")
print(f"Logits: {logits}")
print(f"Etiqueta final: {etiqueta_predicha}")
print(f"Confianza: {score_predicho:.2%}")

resultado_pipeline = clasificador(texto)
print("
La misma predicción usando pipeline:")
print(resultado_pipeline)


---

## 6. Corpus de Prueba: Español Rioplatense

Usamos el mismo corpus de los notebooks anteriores para comparar resultados.

In [ ]:
# Corpus en español rioplatense con expresiones coloquiales
frases = [
    # Positivas
    "La verdad, este lugar está bárbaro. Muy recomendable.",
    "Qué buena onda la atención, volvería sin dudarlo.",
    "Me encantó la comida, aunque la música estaba muy fuerte.",
    "Todo excelente. Atención de diez.",
    "Muy conforme con el resultado final.",
    "Superó mis expectativas, gracias.",
    "El mejor asado que probé en mucho tiempo.",
    "Excelente relación precio-calidad, muy recomendable.",
    "La atención fue impecable, muy atentos.",
    "Me gustó mucho el ambiente tranquilo.",

    # Negativas
    "Una porquería de servicio, nunca más vuelvo.",
    "El envío fue lento y el producto llegó dañado. Qué desastre.",
    "Qué estafa, me arrepiento de haber comprado.",
    "No me gustó para nada la experiencia.",
    "No lo recomiendo, mala calidad.",
    "Malísima atención, el mozo tenía mala onda.",
    "Tardaron dos horas en entregar, llegó todo frío.",
    "Me cobraron de más y encima se hicieron los giles.",
    "La carne estaba pasada, casi no se podía comer.",
    "Pésima experiencia, no vuelvo más.",

    # Adicionales con expresiones argentinas
    "Zafa, pero nada especial.",
    "Está piola el lugar, volvería.",
    "Qué garrón, tardaron una banda.",
    "Re copado todo, la rompieron.",
    "Un bodrio total, no vayan."
]

print(f"Corpus: {len(frases)} frases en español rioplatense")
print(f"\nEjemplos:")
print(f"  [Positiva] {frases[0]}")
print(f"  [Negativa] {frases[10]}")
print(f"  [Coloquial] {frases[20]}")

---

## 7. Clasificación Masiva con Pipelines

Los pipelines pueden procesar listas de textos de forma eficiente.

In [ ]:
print("Clasificando todas las frases con BETO...
")

resultados = clasificador(frases)

print("=" * 70)
print("RESULTADOS DE CLASIFICACIÓN")
print("=" * 70)

for i in range(len(frases)):
    frase = frases[i]
    resultado = resultados[i]
    label = resultado['label']
    score = resultado['score']

    frase_corta = frase
    if len(frase_corta) > 55:
        frase_corta = frase_corta[:52] + "..."

    print(f"
{i + 1:2d}. '{frase_corta}'")
    print(f"    -> {label} (confianza: {score:.2%})")


---

## 8. Análisis de Resultados

Analicemos qué tan bien BETO maneja expresiones coloquiales argentinas.

In [ ]:
frases_argentinas = [
    ("Zafa, pero nada especial.", "NEU o POS"),
    ("Está piola el lugar, volvería.", "POS"),
    ("Qué garrón, tardaron una banda.", "NEG"),
    ("Re copado todo, la rompieron.", "POS"),
    ("Un bodrio total, no vayan.", "NEG"),
    ("Qué buena onda la atención.", "POS"),
    ("Malísima atención, mala onda.", "NEG")
]

print("=" * 70)
print("ANÁLISIS: EXPRESIONES ARGENTINAS")
print("=" * 70)

solo_frases_argentinas = []
for frase, _ in frases_argentinas:
    solo_frases_argentinas.append(frase)

resultados_arg = clasificador(solo_frases_argentinas)

aciertos = 0
for i in range(len(frases_argentinas)):
    frase, esperado = frases_argentinas[i]
    resultado = resultados_arg[i]
    predicho = resultado['label']
    score = resultado['score']

    correcto = predicho in esperado
    if correcto:
        aciertos += 1
        marca = "OK"
    else:
        marca = "REVISAR"

    print(f"
{marca} '{frase}'")
    print(f"  Esperado: {esperado} | Predicho: {predicho} ({score:.2%})")

print("
" + "=" * 70)
print(f"Aciertos aproximados: {aciertos}/{len(frases_argentinas)}")
print("Lectura pedagógica: BETO maneja bastante bien español general, pero la jerga muy localizada puede requerir datos del dominio.")


---

## 9. Comparación con los modelos anteriores

El objetivo de esta comparación no es decir que un enfoque vuelve inútil al otro. Lo que buscamos es ubicar cada herramienta en el lugar correcto dentro de la secuencia didáctica y del trabajo profesional.


In [ ]:
print("=" * 70)
print("COMPARACIÓN: MODELOS ENTRENADOS DESDE CERO VS. TRANSFORMERS")
print("=" * 70)

comparacion = [
    ("Preparación inicial", "Construcción y entrenamiento manual", "Modelo ya preentrenado"),
    ("Cantidad de código", "Mayor", "Menor"),
    ("Datos necesarios", "Más datos para aprender bien", "Menos datos para empezar"),
    ("Comprensión del proceso", "Muy transparente", "Más abstracta"),
    ("Contexto lingüístico", "Limitado", "Mucho más rico"),
    ("Costo computacional", "Bajo a medio", "Más alto en memoria y descarga"),
    ("Interpretabilidad", "Más directa", "Más difícil"),
    ("Uso profesional", "Baseline o tareas simples", "Estándar moderno para muchas tareas")
]

print(f"
{'Aspecto':<28} | {'Modelos anteriores':<32} | {'Transformers':<28}")
print("-" * 95)
for fila in comparacion:
    aspecto = fila[0]
    descripcion_1 = fila[1]
    descripcion_2 = fila[2]
    print(f"{aspecto:<28} | {descripcion_1:<32} | {descripcion_2:<28}")

print("
Conclusión:")
print("- Lo anterior fue indispensable para entender cómo aprende un modelo.")
print("- Transformers reduce mucho trabajo de implementación, pero aumenta la necesidad de criterio al elegir y evaluar modelos.")


---

## 10. Explorando Otros Modelos en Español

HuggingFace tiene múltiples modelos para español. Veamos algunos.

### 10.1. RoBERTuito: especializado en lenguaje de redes

RoBERTuito fue entrenado con datos más cercanos al registro de redes sociales. Por eso suele reaccionar mejor frente a abreviaciones, expresiones informales y giros coloquiales.


In [ ]:
print("Cargando RoBERTuito...
")

clasificador_twitter = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

print("RoBERTuito cargado correctamente.
")

frases_redes = [
    "Jajaja re copado el lugar, volvería.",
    "No, qué garrón, mal servicio.",
    "Diez puntos, recomendadísimo.",
    "Una estafa total, no vayan."
]

resultados_twitter = clasificador_twitter(frases_redes)

for i in range(len(frases_redes)):
    frase = frases_redes[i]
    resultado = resultados_twitter[i]
    print(f"Frase de redes: '{frase}'")
    print(f"  -> {resultado['label']} ({resultado['score']:.2%})
")


### 10.2. Comparación BETO vs. RoBERTuito

Ahora usamos ambos modelos sobre las mismas frases para ver si la diferencia de dominio se vuelve visible.


In [ ]:
print("=" * 70)
print("COMPARACIÓN: BETO vs. RoBERTuito")
print("=" * 70)

frases_test = [
    "Está piola el lugar, re tranquilo.",
    "Qué bodrio, nunca más vuelvo.",
    "La comida estaba bien, nada del otro mundo."
]

resultados_beto = clasificador(frases_test)
resultados_robertuito = clasificador_twitter(frases_test)

for i in range(len(frases_test)):
    frase = frases_test[i]
    resultado_beto = resultados_beto[i]
    resultado_robertuito = resultados_robertuito[i]

    print(f"
Frase: '{frase}'")
    print(f"  BETO:       {resultado_beto['label']:8s} ({resultado_beto['score']:.2%})")
    print(f"  RoBERTuito: {resultado_robertuito['label']:8s} ({resultado_robertuito['score']:.2%})")

print("
Observaciones:")
print("- BETO suele funcionar bien en español general.")
print("- RoBERTuito suele resultar más natural en lenguaje de redes y jerga coloquial.")
print("- La elección depende del dominio real de tu aplicación.")


---

## 11. Casos difíciles: límites del modelo

Aunque el resultado sea fuerte en muchos ejemplos, sigue habiendo zonas problemáticas: ironía, sarcasmo, mezcla de sentimientos y ambigüedad contextual.


In [ ]:
print("=" * 70)
print("CASOS DIFÍCILES: IRONÍA, SARCASMO Y AMBIGÜEDAD")
print("=" * 70)

casos_dificiles = [
    "Buenísimo, justo lo que necesitaba, tardar 3 horas.",
    "Claro, excelente idea cobrarme el doble.",
    "La comida excelente pero el servicio pésimo.",
    "Interesante experiencia.",
    "No estuvo nada mal."
]

resultados_dificiles = clasificador(casos_dificiles)

for i in range(len(casos_dificiles)):
    caso = casos_dificiles[i]
    resultado = resultados_dificiles[i]

    print(f"
Frase: '{caso}'")
    print(f"  BETO dice: {resultado['label']} ({resultado['score']:.2%})")
    print("  Pregunta de lectura: ¿capturó bien el matiz o simplificó demasiado la frase?")

print("
Observaciones generales:")
print("- La ironía sigue siendo muy difícil sin contexto adicional.")
print("- Cuando hay señales mezcladas, el modelo suele forzar una sola etiqueta final.")
print("- Un score alto no garantiza comprensión perfecta del caso.")


---

## 12. Próximos pasos: fine-tuning

Lo que vimos hasta acá fue inferencia con un modelo ya ajustado. El siguiente paso profesional es adaptar ese modelo a un dominio o corpus propio.


### ¿Cuándo conviene hacer fine-tuning?

**Usar el modelo tal como viene:**
- Cuando necesitás un prototipo rápido.
- Cuando la tarea es bastante general.
- Cuando todavía no tenés datos etiquetados propios.

**Hacer fine-tuning:**
- Cuando el dominio es específico.
- Cuando la jerga del corpus se aleja del entrenamiento original.
- Cuando la métrica actual no alcanza para tu caso de uso.
- Cuando ya contás con un conjunto razonable de ejemplos etiquetados.

### Esquema general del proceso

```python
from transformers import AutoModelForSequenceClassification, Trainer

modelo = AutoModelForSequenceClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased"
)

train_dataset = tokenize_and_encode(train_texts, train_labels)
eval_dataset = tokenize_and_encode(eval_texts, eval_labels)

trainer = Trainer(
    model=modelo,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()
```

Más adelante en el curso vamos a trabajar este proceso con mayor detalle y con evaluación más rigurosa.


---

## Guía Teórico-Conceptual

### 1. Arquitectura Transformer: Conceptos Clave

**Attention Mechanism (Self-Attention):**

La innovación central de los transformers. Permite que cada palabra "preste atención" a todas las demás:

```
Frase: "El banco está cerrado"

Self-attention permite que "banco" mire:
  - "El" → Artículo (bajo peso)
  - "está" → Verbo estado (medio peso)
  - "cerrado" → Alto peso! Disambigua "banco" (institución vs. asiento)
```

**Fórmula simplificada:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Donde:
- **Q (Query)**: "¿Qué busco?"
- **K (Key)**: "¿Qué ofrezco?"
- **V (Value)**: "¿Qué información tengo?"

**Multi-Head Attention:**
- Múltiples attention en paralelo
- Cada "cabeza" aprende patrones diferentes
- BERT tiene 12 capas × 12 cabezas = 144 atenciones

**Ventajas sobre LSTM:**
1. **Paralelización**: Todas las palabras se procesan simultáneamente
2. **Dependencias largas**: Cualquier palabra puede atender a cualquier otra
3. **Interpretabilidad**: Los pesos de attention son visualizables

### 2. BERT vs. GPT: Encoder vs. Decoder

| Aspecto | BERT | GPT |
|---------|------|-----|
| Arquitectura | Encoder (bidirectional) | Decoder (unidirectional) |
| Lectura | Lee toda la oración | Lee izquierda a derecha |
| Pre-entrenamiento | Masked LM | Next token prediction |
| Mejor para | Clasificación, NER, QA | Generación de texto |
| Ejemplos | BERT, RoBERTa, BETO | GPT-2, GPT-3, GPT-4 |

**Masked Language Modeling (BERT):**
```
Input:  "El [MASK] está cerrado"
Output: "banco" (con probabilidad 0.8)
```

**Next Token Prediction (GPT):**
```
Input:  "El banco está"
Output: "cerrado" (con probabilidad 0.6)
```

### 3. Tokenización en Transformers

Los transformers usan **sub-word tokenization** (WordPiece, BPE):

**Ventajas sobre tokenización por palabra:**
```
Palabra completa:
"recomendabilísimo" → <UNK> (desconocida)

WordPiece:
"recomendabilísimo" → ["recomienda", "##bili", "##simo"]
```

Esto permite:
- Vocabulario finito (30k tokens)
- Manejar palabras nuevas
- Capturar morfología (prefijos, sufijos)

### 4. Pre-entrenamiento: La Fase Cara

**Corpus de pre-entrenamiento:**
- BERT (inglés): Wikipedia (2.5B palabras) + BookCorpus (800M palabras)
- BETO (español): Wikipedia español (3B palabras)
- GPT-3: Common Crawl (570GB de texto)

**Recursos computacionales:**
- BERT: 4 días en 16 TPUs (~$7k)
- GPT-3: Semanas en clusters masivos (~$12M)

**Por esto NO entrenamos desde cero:**
- Costo prohibitivo
- Expertise técnico alto
- Modelos ya disponibles

### 5. Fine-tuning: La Fase Accesible

**Proceso:**
1. **Congelar capas inferiores**: Mantener conocimiento lingüístico general
2. **Entrenar capas superiores**: Adaptar a tarea específica
3. **Pocos epochs**: 2-5 típicamente
4. **Learning rate bajo**: 1e-5 a 5e-5

**Datos necesarios:**
- Clasificación: 500-1000 muestras
- NER: 1000-5000 entidades
- QA: 500-2000 pares pregunta-respuesta

**Tiempo:**
- GPU T4 (Google Colab): 30-60 minutos
- GPU A100: 5-10 minutos

### 6. HuggingFace Model Hub

**Estadísticas (2025):**
- 500,000+ modelos
- 100+ idiomas
- Todas las arquitecturas: BERT, GPT, T5, LLaMA, etc.

**Modelos destacados para español:**
1. **BETO** (`dccuchile/bert-base-spanish-wwm-cased`)
   - BERT base en español
   - 110M parámetros
   - General purpose

2. **RoBERTuito** (`pysentimiento/robertuito-*`)
   - RoBERTa en tweets español
   - Jerga, emojis, hashtags
   - Mejor para redes sociales

3. **MarIA** (`PlanTL-GOB-ES/roberta-base-bne`)
   - RoBERTa español
   - Corpus BNE (Biblioteca Nacional)
   - Muy robusto

4. **mBERT** (`bert-base-multilingual-cased`)
   - 104 idiomas incluyendo español
   - Útil para tareas multilingües

### 7. Limitaciones de Transformers

**1. Longitud de secuencia:**
- BERT: Máximo 512 tokens
- Documentos largos requieren truncamiento o chunking
- Soluciones: Longformer, BigBird (hasta 4096 tokens)

**2. Recursos computacionales:**
- Modelos grandes (400MB - 2GB)
- Inferencia más lenta que modelos simples
- GPU recomendada para fine-tuning

**3. Interpretabilidad:**
- 110M parámetros son difíciles de interpretar
- Attention weights ayudan pero no explican todo
- "Caja menos negra" que antes, pero aún opaco

**4. Sesgos del pre-entrenamiento:**
- Heredan sesgos del corpus (Wikipedia, internet)
- Requiere cuidado en aplicaciones sensibles
- Fine-tuning puede amplificar sesgos

**5. Overfitting en fine-tuning:**
- Con pocos datos, puede sobreajustar
- Requiere regularización, early stopping
- Data augmentation ayuda

---

## Preguntas y Respuestas para Estudio

### Preguntas Conceptuales

**1. ¿Qué es transfer learning en NLP y por qué es tan importante?**

*Respuesta:* Transfer learning es usar un modelo pre-entrenado en un corpus masivo (Wikipedia, Common Crawl) y adaptarlo a una tarea específica con pocos datos. Es importante porque:
1. El modelo ya "entiende" lenguaje (gramática, semántica)
2. Solo necesitás 500-1000 muestras en vez de millones
3. Ahorras meses de entrenamiento y miles de dólares
4. Obtenés estado del arte sin expertise en deep learning

**2. ¿Cuál es la principal innovación de los transformers sobre LSTM?**

*Respuesta:* **Self-attention mechanism**. En lugar de procesar secuencialmente (palabra por palabra), los transformers permiten que todas las palabras se "vean" entre sí simultáneamente. Esto:
- Permite paralelización (mucho más rápido)
- Captura dependencias largas mejor
- Es más interpretable (visualizar attention weights)

**3. ¿Qué es un pipeline en HuggingFace y qué problema resuelve?**

*Respuesta:* Un pipeline es una abstracción de alto nivel que encapsula:
1. **Tokenización**: Texto → tokens
2. **Modelo**: Tokens → embeddings → predicciones
3. **Post-procesamiento**: Predicciones → formato legible

Resuelve el problema de complejidad: en lugar de manejar manualmente tokenizers, modelos y configuraciones, el pipeline lo hace en una línea de código.

**4. ¿Por qué BERT es "bidirectional" y GPT es "unidirectional"?**

*Respuesta:*
- **BERT**: Lee toda la oración en ambas direcciones. En "El banco está cerrado", "banco" ve tanto "El" (izquierda) como "cerrado" (derecha)
- **GPT**: Solo lee de izquierda a derecha. En "El banco está", solo puede usar "El banco" para predecir "está"

Por eso BERT es mejor para clasificación/comprensión y GPT para generación.

**5. ¿Qué es Masked Language Modeling y por qué es efectivo para pre-entrenamiento?**

*Respuesta:* MLM oculta aleatoriamente el 15% de las palabras y el modelo debe predecirlas: "El [MASK] está cerrado" → "banco". Es efectivo porque:
- Fuerza al modelo a entender contexto bidireccional
- Es auto-supervisado (no requiere etiquetas humanas)
- Aprende representaciones profundas del lenguaje

### Preguntas Técnicas

**6. En el código, ¿qué hace exactamente `pipeline("sentiment-analysis", model="...")` internamente?**

*Respuesta:*
1. Descarga el modelo y tokenizer desde HuggingFace Hub (si no está en caché)
2. Carga el modelo pre-entrenado en memoria
3. Configura el tokenizer apropiado (WordPiece para BERT)
4. Crea un pipeline que encadena: tokenización → modelo → argmax → label

**7. ¿Por qué los modelos transformer son tan grandes (400MB+)?**

*Respuesta:*
- BERT base: 110M parámetros × 4 bytes (float32) = 440MB
- Cada parámetro es un peso de la red neuronal
- 12 capas × 12 attention heads × embeddings × FFN = muchos parámetros

Comparado con nuestro MLP (280 parámetros = 1KB), es ~400,000 veces más grande.

**8. ¿Qué significa el "score" en el resultado del pipeline?**

*Respuesta:* Es la probabilidad (softmax) que el modelo asigna a esa clase:
- score=0.95 → 95% seguro de la predicción
- score=0.55 → Apenas seguro (predicción incierta)

Útil para:
- Filtrar predicciones inciertas (threshold > 0.7)
- Priorizar casos para revisión humana

**9. ¿Cuál es la diferencia entre `model` y `pipeline` en HuggingFace?**

*Respuesta:*
- **Pipeline**: API de alto nivel, maneja todo automáticamente, fácil de usar
- **Model**: API de bajo nivel, control total, requiere manejar tokenización manualmente

```python
# Pipeline (simple)
resultado = pipeline("sentiment-analysis")("texto")

# Model (control manual)
tokenizer = AutoTokenizer.from_pretrained("beto")
model = AutoModel.from_pretrained("beto")
inputs = tokenizer("texto", return_tensors="pt")
outputs = model(**inputs)
```

**10. ¿Por qué BETO y RoBERTuito dan resultados diferentes en la misma frase?**

*Respuesta:*
1. **Corpus de pre-entrenamiento diferente**:
   - BETO: Wikipedia (formal)
   - RoBERTuito: Twitter (informal)
2. **Fine-tuning diferente**: Diferentes datos de sentiment analysis
3. **Vocabulario**: RoBERTuito conoce mejor jerga, emojis

La elección depende del dominio de tu aplicación.

### Preguntas de Aplicación

**11. Tenés 200 reseñas etiquetadas de un restaurante. ¿Usarías el modelo as-is o harías fine-tuning?**

*Respuesta:* **Depende:**
- **As-is si**: El modelo out-of-the-box ya da >85% accuracy, el dominio es general
- **Fine-tuning si**:
  - Necesitás >90% accuracy
  - Hay vocabulario muy específico del restaurante
  - 200 muestras son borderline (mínimo recomendado es 500), pero podés intentar con data augmentation

**12. ¿Cómo manejarías un documento de 2000 palabras con BERT (límite: 512 tokens)?**

*Respuesta:* Opciones:
1. **Truncamiento**: Tomar primeros 512 tokens (pierde info del final)
2. **Chunking + agregación**: Dividir en chunks de 512, clasificar cada uno, agregar (voting, promedio)
3. **Extractivo**: Identificar sección clave (ej: resumen, intro) y clasificar eso
4. **Modelo especializado**: Usar Longformer o BigBird (hasta 4096 tokens)

**13. En producción, necesitás clasificar 10,000 textos/segundo. ¿Transformers son apropiados?**

*Respuesta:* Probablemente **no** para latencia ultra-baja:
- BERT inference: ~50-100ms/texto en GPU
- 10k/seg = 0.1ms/texto (100x más rápido requerido)

**Soluciones:**
1. **Distillation**: DistilBERT (60% más rápido, 97% accuracy)
2. **Quantization**: INT8 en vez de FP32 (4x más rápido)
3. **Caching**: Cachear predicciones para textos comunes
4. **Ensemble**: Modelo simple (Naive Bayes) para mayoría, BERT solo para casos inciertos

**14. ¿Cómo usarías HuggingFace para un chatbot que genera respuestas?**

*Respuesta:*
```python
# Usar modelo generativo (GPT)
from transformers import pipeline

generador = pipeline("text-generation", model="gpt2")

respuesta = generador(
    "Usuario: Hola, ¿cómo estás?\nBot:",
    max_length=50,
    num_return_sequences=1
)
```

Para español, usar modelos como `DeepESP/gpt2-spanish`.

**15. Diseñá un sistema de moderación de contenido con transformers.**

*Respuesta:*
```python
# Pipeline multimodelo

# 1. Clasificación de toxicidad
toxicidad = pipeline("text-classification", model="modelo-toxicidad-es")

# 2. Detección de spam
spam = pipeline("text-classification", model="modelo-spam-es")

# 3. NER para detectar info personal
ner = pipeline("ner", model="modelo-ner-es")

def moderar(texto):
    # Análisis paralelo
    es_toxico = toxicidad(texto)[0]['score'] > 0.7
    es_spam = spam(texto)[0]['score'] > 0.8
    tiene_pii = any(ent['entity'] == 'PER' for ent in ner(texto))
    
    if es_toxico or es_spam:
        return "RECHAZAR"
    elif tiene_pii:
        return "ADVERTIR"
    else:
        return "APROBAR"
```

---

## Ejercicios propuestos

### Ejercicio 1: explorar el Model Hub
Buscá tres modelos para español y registrá para qué tarea fue ajustado cada uno.

### Ejercicio 2: análisis de confianza
Tomá varias frases y compará cuáles reciben scores altos y cuáles quedan cerca del umbral.

### Ejercicio 3: BETO vs. RoBERTuito
Armá un conjunto chico de frases etiquetadas manualmente y compará ambos modelos con una métrica simple.

### Ejercicio 4: NER con transformers
Probá un pipeline de reconocimiento de entidades y observá qué cambia cuando la tarea ya no es clasificación de sentimientos.

### Ejercicio 5: errores diseñados a mano
Escribí frases con ironía, negaciones dobles o señales mixtas y analizá dónde falla cada modelo.

## Cierre

Este notebook marcó el paso desde modelos entrenados en clase hacia modelos reutilizados desde la práctica profesional. El cambio principal no fue solo técnico: también cambió el tipo de criterio que necesitamos para trabajar bien.

En el próximo cuaderno vamos a usar representaciones semánticas a nivel de oración para búsqueda, similitud y detección de paráfrasis.
